# Big Data: Trabajo Practico 1 - Grupo 5

In [ ]:
# importamos paquetes
import pandas as pd
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# ---------------------------------------------------------------
# Rutas del proyecto. El notebook vive en programs/ y lee de
# ../input/ y escribe en ../output/. Funciona tanto si se ejecuta
# desde programs/ como desde la raiz del TP.
# ---------------------------------------------------------------
from pathlib import Path

BASE = Path.cwd() if (Path.cwd() / "input").is_dir() else Path.cwd().parent
INPUT = BASE / "input"
OUTPUT = BASE / "output"
OUTPUT.mkdir(exist_ok=True)

print("Input :", INPUT)
print("Output:", OUTPUT)

## PARTE I: Familiarizandonos con la base EPH

### Ejercicio 2: bases de datos

In [ ]:
df1 = pd.read_excel(INPUT / "usu_individual_T124.xlsx")
df2 = pd.read_stata(INPUT / "usu_individual_T104.dta", convert_categoricals=False)

#### a) eliminar datos que no son de la region Cuyo (42) y crear base unida

In [ ]:
# dejamos solo datos de Cuyo
df1 = df1[df1["REGION"] == 42]
df2 = df2[df2["region"] == 42]

In [ ]:
# para que las columnas se llamen igual, cambiamos los nombres de las columnas de la segunda base a mayuscula
df2.columns = df2.columns.str.upper()

In [ ]:
# convertimos las columnas en formato incorrecto en df2 a numerico
formato_incorrecto = ['CH14', 'CH15_COD', 'CH16_COD', 'PP04B_COD', 'PP04D_COD', 'PP11B_COD', 'PP11D_COD', 'DECOCUR', 'IDECOCUR', 'RDECOCUR', 'GDECOCUR', 'PDECOCUR', 'ADECOCUR', 'DECINDR', 'IDECINDR', 'RDECINDR', 'GDECINDR', 'PDECINDR', 'ADECINDR', 'DECIFR', 'IDECIFR', 'RDECIFR', 'GDECIFR', 'PDECIFR', 'ADECIFR', 'DECCFR', 'IDECCFR', 'RDECCFR', 'GDECCFR', 'PDECCFR', 'ADECCFR']
for col in formato_incorrecto:
    df2[col] = pd.to_numeric(df2[col], errors='coerce')

In [ ]:
# hacemos un merge vertical
df = pd.concat([df1, df2], ignore_index=True)

#### b) heatmap para las siguientes 10 variables: 

In [ ]:
# definimos las 10 variables de interés
cols_interest = ['CH14', 'CH15_COD', 'CH16_COD', 'PP03C', 'PP03I', 'PP04B_COD', 'PP06H', 'PP07A', 'PP10A', 'PP11A']

In [ ]:
# calculamos porcentaje de datos faltantes
percentage_data_before = (df.groupby("ANO4")[cols_interest].apply(lambda x: x.notnull().mean() * 100))

In [ ]:
# graficamos
plt.figure(figsize=(10, 6))
sns.heatmap(percentage_data_before.T, annot=True, fmt=".3f", cmap="coolwarm_r")
plt.title("Porcentaje de datos no nulos por año (antes de la limpieza)")
plt.xlabel("Año")
plt.savefig(OUTPUT / "Heatmap 1.png", dpi=300, bbox_inches='tight')
plt.show()

#### c) eliminamos observaciones sin sentido

In [ ]:
# en la columna CH06 (edad), hay unas observaciones con valor -1, que deberia representar "Menor de 1", convertimos a 0
df['CH06'] = df['CH06'].astype(float).replace(-1.0, 0)

In [ ]:
# la columna CH14 (último año que aprobó) tiene valores que no corresponden (98) en la base 2004 (existe en 2024)
df.loc[df['ANO4'] == 2004, 'CH14'] = df.loc[df['ANO4'] == 2004, 'CH14'].astype(float).replace(98.0, float("NaN"))

In [ ]:
# la columna PP06C (ingresos) tiene valores negativos que no son -9 (ns/nc)
df['PP06C'] = df['PP06C'].astype(float).replace(-8.0, float('NaN'))
df['PP06C'] = df['PP06C'].astype(float).replace(-7.0, float('NaN'))

In [ ]:
# la columna PP06D (ingresos) tiene valores negativos que no son -9 (ns/nc)
df['PP06D'] = df['PP06D'].astype(float).replace(-8.0, float('NaN'))
df['PP06D'] = df['PP06D'].astype(float).replace(-7.0, float('NaN'))

#### d) limpieza de datos

In [ ]:
# buscamos las columnas que existan solo en una de las bases
columnas_unicas = set(df1.columns).symmetric_difference(set(df2.columns))

In [ ]:
# eliminamos las columnas que existen solo en una de las bases
df = df.drop(columns=[col for col in columnas_unicas if col in df.columns])

In [ ]:
# convertimos MAS_500 en una variable binaria que toma valor 1 si el conglomerado tiene mas de 500 mil habitantes (S) y 0 si no (N)
df['MAS_500'] = df['MAS_500'].astype(str).replace('S', 1)
df['MAS_500'] = df['MAS_500'].astype(str).replace('N', 0)
df['MAS_500'] = pd.to_numeric(df['MAS_500'], errors='coerce')

In [ ]:
# si la persona es menor a dos años (CH09 toma valor 3), entonces nunca aprobó ningún nivel
df.loc[df['CH09'] == 3, 'CH14'] = df.loc[df['CH09'] == 3, 'CH14'].astype(float).replace(float("NaN"), 0.0)

In [ ]:
# si la persona nunca asistió a un establecimiento educativo (CH10 toma valor 3), entonces nunca aprobó ningún nivel
df.loc[df['CH10'] == 3, 'CH14'] = df.loc[df['CH10'] == 3, 'CH14'].astype(float).replace(float("NaN"), 0.0)

In [ ]:
# la columna CH15_COD especifica cosas de la columna CH15, pero deberia tomar valor solo si CH15 es 3, 4 o 5
df.loc[~df['CH15'].isin([3.0, 4.0, 5.0]), 'CH15_COD'] = 0

In [ ]:
# la columna CH16_COD especifica cosas de la columna CH16, pero deberia tomar valor solo si CH16 es 3, 4 o 5
df.loc[~df['CH16'].isin([3.0, 4.0, 5.0]), 'CH16_COD'] = 0

In [ ]:
# la columna PP04B_COD especifica cosas de la columna PP04A, pero deberia tomar valor solo si PP04A es 3
df.loc[df['PP04A'] != 3.0, 'PP04B_COD'] = 0

In [ ]:
# la columna PP06H especifica cosas de la columna PP06E, pero deberia tomar valor solo si PP06E es 2 o 3
df.loc[~df['PP06E'].isin([2.0, 3.0]), 'PP06H'] = 0

In [ ]:
# convertir PP05F (para quien trabaja habitualmente el negocio) en binaria, reemplazar en 1 (6, un sólo cliente) y 2 (7, distintos clientes)
df['PP05F'] = df['PP05F'].astype(float).replace(6.0, 1)
df['PP05F'] = df['PP05F'].astype(float).replace(7.0, 2)

In [ ]:
# convertir PP07F5 (no recibe ningun beneficio) para que tome valor 1 si la respuesta es Si (ahora es 5)
df['PP07F5'] = df['PP07F5'].astype(float).replace(5.0, 1)

In [ ]:
# convertir PP07G59 (no tiene ningun beneficio) para que tome valor 1 si la respuesta es Si (ahora es 5)
df['PP07G_59'] = df['PP07G_59'].astype(float).replace(5.0, 1)

In [ ]:
# reemplazamos los missing values de las columnas que solo aplican a los ocupados si los individuos no son ocupados (0 representa no aplica)
columnas_ocupados = ['PP03C', 'PP03D', 'PP3E_TOT', 'PP3F_TOT', 'PP03G', 'PP03H', 'PP03I', 'PP03J', 'INTENSI', 'PP04A', 'PP04B_COD', 'PP04B1', 'PP04B2', 'PP04B3_MES', 'PP04B3_ANO', 'PP04B3_DIA', 'PP04C', 'PP04C99', 'PP04D_COD', 'PP04G', 'PP05B2_MES', 'PP05B2_ANO', 'PP05B2_DIA', 'PP05C_1', 'PP05C_2', 'PP05C_3', 'PP05E', 'PP05F', 'PP05H', 'PP06A', 'PP06E', 'PP06H', 'PP07A', 'PP07C', 'PP07D', 'PP07E', 'PP07F1', 'PP07F2', 'PP07F3', 'PP07F4', 'PP07F5', 'PP07G1', 'PP07G2', 'PP07G3', 'PP07G4', 'PP07G_59', 'PP07H', 'PP07I', 'PP07J', 'PP07K']

for col in columnas_ocupados:
    df.loc[df['ESTADO'] != 1, col] = df.loc[df['ESTADO'] != 1, col].astype(float).replace(float("NaN"), 0.0)

In [ ]:
# reemplazamos los missing values de las columnas que solo aplican a los desocupados si los individuos no son desocupados (0 representa no aplica)
columnas_desocupados = ['PP10A', 'PP10C', 'PP10D', 'PP10E', 'PP11A', 'PP11B_COD', 'PP11B1', 'PP11B2_MES', 'PP11B2_ANO', 'PP11B2_DIA', 'PP11C', 'PP11C99', 'PP11D_COD', 'PP11G_ANO', 'PP11G_MES', 'PP11G_DIA', 'PP11L', 'PP11L1', 'PP11N', 'PP11O', 'PP11P', 'PP11Q', 'PP11R', 'PP11S', 'PP11T']

for col in columnas_desocupados:
    df.loc[df['ESTADO'] != 2, col] = df.loc[df['ESTADO'] != 2, col].astype(float).replace(float("NaN"), 0.0)

In [ ]:
# eliminamos columnas no necesarias: IMPUTA (no tiene observaciones), PP09A y PP09A_ESP (son solo para CABA), 
# PP09B (es solo para otras provincias, no incluye Cuyo), PP09C y PP09C_ESP (son dependientes de las tres anteriores)
eliminar = ['IMPUTA', 'PP09A', 'PP09A_ESP', 'PP09B', 'PP09C', 'PP09C_ESP'] 
df = df.drop(columns=[col for col in eliminar if col in df.columns])

In [ ]:
# convertimos columnas que son float a int si deberían serlo, dejamos de lado las variables de monto y CODUSU (codigo para distinguir viviendas)
columnas_float = ['PP06C', 'PP06D', 'PP08D1', 'PP08D4', 'PP08F1', 'PP08F2', 'PP08J1', 'PP08J2', 'PP08J3', 'P21', 'TOT_P12', 'P47T', 'V2_M', 'V3_M', 'V4_M', 'V5_M', 'V8_M', 'V9_M', 'V10_M', 'V11_M', 'V12_M', 'V18_M', 'V19_AM', 'V21_M', 'T_VI', 'ITF', 'IPCF']
for col in df.columns:
    if col not in columnas_float and col != "CODUSU":
        df[col] = df[col].astype('Int64')

In [ ]:
# nos aseguramos que el formato de las columnas float este correcto
for col in columnas_float:
    df[col] = df[col].astype('Float64')

#### e) segundo heatmap

In [ ]:
# calculamos porcentaje de datos faltantes después de la limpieza
percentage_data_after = (df.groupby("ANO4")[cols_interest].apply(lambda x: x.notnull().mean() * 100))

In [ ]:
# graficamos
plt.figure(figsize=(10, 6))
sns.heatmap(percentage_data_after.T,
    annot=True, fmt=".3f", cmap="coolwarm_r")
plt.title("Porcentaje de datos no nulos por año (después de la limpieza)")
plt.xlabel("Año")
plt.savefig(OUTPUT / "Heatmap 2.png", dpi=300, bbox_inches='tight')
plt.show()

## PARTE II: Primer Analisis Exploratorio

### Ejercicio 3: grafico de barras

In [ ]:
# agrupamos los datos
datos_sexo = df.groupby(['ANO4', 'CH04']).size().unstack()

In [ ]:
# graficamos
datos_sexo.plot(kind='bar', figsize=(7, 5), color=['royalblue', 'lightpink'])
plt.ylabel('Cantidad de Personas')
plt.xlabel('Año')
plt.title('Composición por Sexo en 2004 y 2024')
plt.xticks(rotation=0)
plt.legend(title="Sexo", labels=['Hombres', 'Mujeres'])
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.savefig(OUTPUT / "Grafico barras 1.png", dpi=300, bbox_inches='tight')
plt.show()

### Ejercicio 4: matriz de correlacion

In [ ]:
# graficamos
variables = ['CH04', 'CH06', 'CH07', 'CH08', 'NIVEL_ED', 'ESTADO', 'CAT_INAC', 'IPCF']

for year in [2004, 2024]:
    plt.figure(figsize=(8, 6))
    datos = df[df['ANO4'] == year][variables]
    corr_matrix = datos.corr()
    
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)

    plt.title(f'Matriz de Correlación ({year})')
    plt.savefig(OUTPUT / f"Matriz {year}.png", dpi=300, bbox_inches='tight')
    plt.show()

## PARTE III: Conociendo a los ocupados y desocupados

### Ejercicio 5: cantidad desocupados e inactivos, IPCF segun estado

In [ ]:
# cantidad de desocupados
cantidad_desocupados = df[df['ESTADO'] == 2].shape[0]
print(f"Hay {cantidad_desocupados} desocupados en la muestra.")

In [ ]:
# cantidad de inactivos
cantidad_inactivos = df[(df['ESTADO'] == 3) | (df['ESTADO'] == 4)].shape[0]
print(f"Hay {cantidad_inactivos} inactivos en la muestra.")

In [ ]:
# media de ingreso per cápita familiar según estado
mean_ocupados = df.loc[df['ESTADO'] == 1, 'IPCF'].mean()
mean_desocupados = df.loc[df['ESTADO'] == 2, 'IPCF'].mean()
mean_inactivos = df.loc[(df['ESTADO'] == 3) | (df['ESTADO'] == 4), 'IPCF'].mean()

print(f"La media de ingreso per cápita familiar para los ocupados es: {mean_ocupados.round(2)}.")
print(f"La media de ingreso per cápita familiar para los desocupados es: {mean_desocupados.round(2)}.")
print(f"La media de ingreso per cápita familiar para los inactivos es: {mean_inactivos.round(2)}.")

### Ejercicio 6: bases respondieron y norespondieron

### Exportamos la base limpia

In [ ]:
# exportamos la base limpia: es el insumo del TP3
df.to_csv(OUTPUT / "base.csv", index=False)
print("base.csv guardada:", df.shape)

In [ ]:
# creamos las bases
respondieron = pd.DataFrame(df[df['ESTADO'] != 0])
norespondieron = pd.DataFrame(df[df['ESTADO'] == 0])

n_no_respondieron = norespondieron.shape[0]
print(f"{n_no_respondieron} personas en la muestra no respondieron sobre su condición de actividad.")

### Ejercicio 7: grafico barras PEA

In [ ]:
# creamos la columna población economicamente activa
respondieron['PEA'] = ((respondieron['ESTADO'] == 1) | (respondieron['ESTADO'] == 2)).astype(int)

In [ ]:
# agrupamos los datos
datos_pea = respondieron.groupby(['ANO4', 'PEA']).size().unstack()

In [ ]:
# graficamos
datos_pea.plot(kind='bar', figsize=(7, 5), color=['coral', 'lightseagreen'])
plt.ylabel('Cantidad de Personas')
plt.xlabel('Año')
plt.title('Distribución Población Económicamente Activa')
plt.xticks(rotation=0)
plt.legend(labels=['No Activa', 'Activa'])
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.savefig(OUTPUT / "Grafico barras 2.png", dpi=300, bbox_inches='tight')
plt.show()

### Ejercicio 8: grafico barras PET

In [ ]:
# creamos la columna población en edad para trabajar
respondieron['PET'] = ((respondieron['CH06'] >= 15) & (respondieron['CH06'] <= 65)).astype(int)

In [ ]:
# agrupamos los datos
datos_pet = respondieron.groupby(['ANO4', 'PET']).size().unstack()

In [ ]:
# graficamos
datos_pet.plot(kind='bar', figsize=(7, 5), color=['coral', 'lightseagreen'])
plt.ylabel('Cantidad de Personas')
plt.xlabel('Año')
plt.title('Distribución Población en Edad para Trabajar')
plt.xticks(rotation=0)
plt.legend(labels=['No Edad Laboral', 'Edad Laboral'])
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.savefig(OUTPUT / "Grafico barras 3.png", dpi=300, bbox_inches='tight')
plt.show()

### Ejercicio 9: desocupados

In [ ]:
# creamos columna desocupados
respondieron['DESOCUPADOS'] = (respondieron['ESTADO'] == 2).astype(int)
desocupacion_2004 = respondieron.loc[respondieron['ANO4'] == 2004, 'DESOCUPADOS'].sum()
desocupacion_2024 = respondieron.loc[respondieron['ANO4'] == 2024, 'DESOCUPADOS'].sum()
print(f'Hubo {desocupacion_2004} desocupados en 2004.')
print(f'Hubo {desocupacion_2024} desocupados en 2024.')

#### a) proporcion de desocupados por nivel educativo

In [ ]:
# buscamos los datos de desocupación por nivel educativo
niveles = ['Primaria Incompleta', 'Primaria Completa', 'Secundaria Incompleta', 'Secundaria Completa', 'Universitaria Incompleta', 'Universitaria Completa', 'Sin instrucción']
desocupados_por_nivel_2004 = []
desocupados_por_nivel_2024 = []
personas_por_nivel_2004 = []
personas_por_nivel_2024 = []

for nivel in range(1, 8):
    desocupados_por_nivel_2004.append(respondieron.loc[(respondieron['NIVEL_ED'] == nivel) & (respondieron['ANO4'] == 2004), 'DESOCUPADOS'].sum())
    personas_por_nivel_2004.append(respondieron.loc[(respondieron['NIVEL_ED'] == nivel) & (respondieron['ANO4'] == 2004), 'PEA'].sum())

for nivel in range(1, 8):
    desocupados_por_nivel_2024.append(respondieron.loc[(respondieron['NIVEL_ED'] == nivel) & (respondieron['ANO4'] == 2024), 'DESOCUPADOS'].sum())
    personas_por_nivel_2024.append(respondieron.loc[(respondieron['NIVEL_ED'] == nivel) & (respondieron['ANO4'] == 2024), 'PEA'].sum())

In [ ]:
# calculamos la proporción
proporcion_nivel_2004 = []
proporcion_nivel_2024 = []

for nivel, desocupados, personas in zip(niveles, desocupados_por_nivel_2004, personas_por_nivel_2004):
    proporcion_nivel_2004.append(desocupados / personas * 100)
    print(f"El {(desocupados / personas * 100).round(2)}% de las personas económicamente activas en la muestra con nivel educativo {nivel} eran desocupados en 2004.")
    
for nivel, desocupados, personas in zip(niveles, desocupados_por_nivel_2024, personas_por_nivel_2024):
    proporcion_nivel_2024.append(desocupados / personas * 100)
    print(f"El {(desocupados / personas * 100).round(2)}% de las personas económicamente activas en la muestra con nivel educativo {nivel} eran desocupados en 2024.")

In [ ]:
personas_por_nivel_2004

In [ ]:
# acomodamos datos para graficar
datos_nivel = pd.DataFrame({
    'Categoria': niveles,
    '2004': proporcion_nivel_2004,
    '2024' : proporcion_nivel_2024})

datos_nivel.set_index('Categoria', inplace=True)

In [ ]:
# graficamos
datos_nivel.plot(kind='bar', figsize=(16, 5), color=['coral', 'lightseagreen'])
plt.ylabel('Porcentaje de Desocupación')
plt.title('Proporción de desocupados por nivel educativo (2004 vs 2024)')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.savefig(OUTPUT / "Grafico barras 4.png", dpi=300, bbox_inches='tight')
plt.show()

#### b) variable categórica de años cumplidos (CH06)

In [ ]:
# agrupamos por edad
respondieron['GRUPO_ETARIO'] = pd.cut(df['CH06'], bins=range(0, 101, 10), right=False, labels=[i for i in range(0, 100, 10)])

In [ ]:
# buscamos los datos de desocupación por edad
edades = [f"{i} a {i+9}" for i in range(0, 100, 10)]
desocupados_por_edad_2004 = []
desocupados_por_edad_2024 = []
personas_por_edad_2004 = []
personas_por_edad_2024 = []

for edad in range(0, 101, 10):
    desocupados_por_edad_2004.append(respondieron.loc[(respondieron['GRUPO_ETARIO'] == edad) & (respondieron['ANO4'] == 2004), 'DESOCUPADOS'].sum())
    personas_por_edad_2004.append(respondieron.loc[(respondieron['GRUPO_ETARIO'] == edad) & (respondieron['ANO4'] == 2004), 'PEA'].sum())

for edad in range(0, 101, 10):
    desocupados_por_edad_2024.append(respondieron.loc[(respondieron['GRUPO_ETARIO'] == edad) & (respondieron['ANO4'] == 2024), 'DESOCUPADOS'].sum())
    personas_por_edad_2024.append(respondieron.loc[(respondieron['GRUPO_ETARIO'] == edad) & (respondieron['ANO4'] == 2024), 'PEA'].sum())

In [ ]:
# calculamos la proporción
proporcion_edad_2004 = []
proporcion_edad_2024 = []

for edad, desocupados, personas in zip(edades, desocupados_por_edad_2004, personas_por_edad_2004):
    if personas > 0:
        proporcion_edad_2004.append(desocupados / personas * 100)
        print(f"El {(desocupados / personas * 100).round(2)}% de las personas económicamente activas en la muestra del grupo etario {edad} eran desocupados en 2004.")
    else:
        proporcion_edad_2004.append(0.0)
        print(f"No hay personas económicamente activas en el grupo etario {edad} en 2004.")

for edad, desocupados, personas in zip(edades, desocupados_por_edad_2024, personas_por_edad_2024):
    if personas > 0:
        proporcion_edad_2024.append(desocupados / personas * 100)
        print(f"El {(desocupados / personas * 100).round(2)}% de las personas económicamente activas en la muestra del grupo etario {edad} eran desocupados en 2024.")
    else:
        proporcion_edad_2024.append(0.0)
        print(f"No hay personas económicamente activas en el grupo etario {edad} en 2024.")

In [ ]:
# acomodamos datos para graficar
datos_edad = pd.DataFrame({
    'Categoria': edades,
    '2004': proporcion_edad_2004,
    '2024' : proporcion_edad_2024})

datos_edad.set_index('Categoria', inplace=True)

In [ ]:
# graficamos
datos_edad.plot(kind='bar', figsize=(9, 5), color=['coral', 'lightseagreen'])
plt.ylabel('Porcentaje de Desocupación')
plt.title('Proporción de desocupados por edad (2004 vs 2024)')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.savefig(OUTPUT / "Grafico barras 5.png", dpi=300, bbox_inches='tight')
plt.show()

#### c) diferencias de nivel (2004 vs 2024)

In [ ]:
# buscamos los datos de desocupación por estado conyugal
estados = ["Unido", "Casado", "Separado/a ó divorciado/a", "Viudo/a", "Soltero/a"]
desocupados_por_estado_2004 = []
desocupados_por_estado_2024 = []
personas_por_estado_2004 = []
personas_por_estado_2024 = []

for estado in range(1, 6):
    desocupados_por_estado_2004.append(respondieron.loc[(respondieron['CH07'] == estado) & (respondieron['ANO4'] == 2004), 'DESOCUPADOS'].sum())
    personas_por_estado_2004.append(respondieron.loc[(respondieron['CH07'] == estado) & (respondieron['ANO4'] == 2004), 'PEA'].sum())

for estado in range(1, 6):
    desocupados_por_estado_2024.append(respondieron.loc[(respondieron['CH07'] == estado) & (respondieron['ANO4'] == 2024), 'DESOCUPADOS'].sum())
    personas_por_estado_2024.append(respondieron.loc[(respondieron['CH07'] == estado) & (respondieron['ANO4'] == 2024), 'PEA'].sum())

In [ ]:
# calculamos la proporción
proporcion_estado_2004 = []
proporcion_estado_2024 = []

for estado, desocupados, personas in zip(estados, desocupados_por_estado_2004, personas_por_estado_2004):
    proporcion_estado_2004.append(desocupados / personas * 100) 
    print(f"El {(desocupados / personas * 100).round(2)}% de las personas económicamente activas en la muestra de la categoría {estado} eran desocupados en 2004.")

for estado, desocupados, personas in zip(estados, desocupados_por_estado_2024, personas_por_estado_2024):
    proporcion_estado_2024.append(desocupados / personas * 100) 
    print(f"El {(desocupados / personas * 100).round(2)}% de las personas económicamente activas en la muestra de la categoría {estado} eran desocupados en 2024.")

In [ ]:
# acomodamos datos para graficar
datos_estados = pd.DataFrame({
    'Categoria': estados,
    '2004': proporcion_estado_2004,
    '2024' : proporcion_estado_2024})

datos_estados.set_index('Categoria', inplace=True)

In [ ]:
# graficamos
datos_estados.plot(kind='bar', figsize=(9, 5), color=['coral', 'lightseagreen'])
plt.ylabel('Porcentaje de Desocupación')
plt.title('Proporción de desocupados por estado conyugal (2004 vs 2024)')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.savefig(OUTPUT / "Grafico barras 6.png", dpi=300, bbox_inches='tight')
plt.show()

### Ejercicio 10: dos tasas de desocupación

In [ ]:
# tasa según el INDEC
tasa_indec_2004 = desocupacion_2004 / respondieron['PEA'].sum() * 100
tasa_indec_2024 = desocupacion_2024 / respondieron['PEA'].sum() * 100
print(f"La tasa de desocupados según el INDEC fue {tasa_indec_2004.round(2)}% en 2004.")
print(f"La tasa de desocupados según el INDEC fue {tasa_indec_2024.round(2)}% en 2024.")

In [ ]:
# tasa alternativa
tasa_alternativa_2004 = desocupacion_2004 / respondieron['PET'].sum() * 100
tasa_alternativa_2024 = desocupacion_2024 / respondieron['PET'].sum() * 100
print(f"La tasa de desocupados alternativa fue {tasa_alternativa_2004.round(2)}% en 2004.")
print(f"La tasa de desocupados alternativa fue {tasa_alternativa_2024.round(2)}% en 2024.")

In [ ]:
# agrupamos datos para graficar
datos_tasas = pd.DataFrame({
    'Categoria': ['Tasa INDEC', 'Tasa Alternativa'],
    '2004': [tasa_indec_2004, tasa_alternativa_2004],
    '2024' : [tasa_indec_2024, tasa_alternativa_2024]})

datos_tasas.set_index('Categoria', inplace=True)

In [ ]:
# graficamos
datos_tasas.plot(kind='bar', figsize=(6, 5), color=['coral', 'lightseagreen'])
plt.ylabel('Tasa de Desocupación')
plt.title('Tasas de Desocupación (2004 vs 2024)')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.savefig(OUTPUT / "Grafico barras 7.png", dpi=300, bbox_inches='tight')
plt.show()

FIN :)